# Notebook 4 — Delta Lake Storage (Kafka → MinIO)
**Project**: Real-Time Retail Analytics & Demand Prediction Platform  
**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar  
**Stack**: Kafka → Pandas → Delta Lake on MinIO (`retail-v2` bucket)  

Consumes records from Kafka topic `retail-txn-v2`, engineers features, writes Delta tables to MinIO.

---

In [7]:
import sys
!{sys.executable} -m pip install deltalake pyarrow kafka-python boto3 -q
print('Done.')

Done.


## 4.1 Config

In [8]:
import pandas as pd
import numpy as np
import json
import time
import gc
import pyarrow as pa
import warnings
warnings.filterwarnings('ignore')
from deltalake.writer import write_deltalake
from deltalake import DeltaTable

# Kafka
KAFKA_BROKERS = 'kafka:9092'
TOPIC         = 'retail-txn-v2'

# MinIO — retail-v2 bucket (created in NB1)
STORAGE = {
    'AWS_ENDPOINT_URL':           'http://minio:9000',
    'AWS_ACCESS_KEY_ID':          'admin',
    'AWS_SECRET_ACCESS_KEY':      'bigdata123',
    'AWS_ALLOW_HTTP':             'true',
    'AWS_S3_ALLOW_UNSAFE_RENAME': 'true',
    'AWS_REGION':                 'us-east-1'
}

# Delta paths on retail-v2 bucket
DELTA_TX   = 's3://retail-v2/delta/transactions'
DELTA_FEAT = 's3://retail-v2/delta/features'
DELTA_AGG  = 's3://retail-v2/delta/aggregations'

print('Config loaded.')
print(f'Kafka: {KAFKA_BROKERS} → Topic: {TOPIC}')
print(f'Delta: retail-v2 bucket on MinIO')

Config loaded.
Kafka: kafka:9092 → Topic: retail-txn-v2
Delta: retail-v2 bucket on MinIO


## 4.2 Consume from Kafka

In [9]:
from kafka import KafkaConsumer

MAX_CONSUME = 500_000

print(f'Consuming up to {MAX_CONSUME:,} records from Kafka...\n')

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKERS,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=15000,
    group_id='delta-writer-v2',
    max_poll_records=2000
)

records = []
start = time.time()

for msg in consumer:
    records.append(msg.value)
    if len(records) % 50_000 == 0:
        elapsed = time.time() - start
        print(f'  Read {len(records):>8,} | {len(records)/elapsed:>6.0f} rec/s')
    if len(records) >= MAX_CONSUME:
        break

consumer.close()
total_time = time.time() - start
print(f'\n✅ Consumed {len(records):,} records in {total_time:.1f}s')

Consuming up to 500,000 records from Kafka...

  Read   50,000 |  11358 rec/s
  Read  100,000 |  17219 rec/s
  Read  150,000 |  20446 rec/s
  Read  200,000 |  22439 rec/s
  Read  250,000 |  23623 rec/s
  Read  300,000 |  24467 rec/s
  Read  350,000 |  24469 rec/s
  Read  400,000 |  24352 rec/s
  Read  450,000 |  24384 rec/s
  Read  500,000 |  24797 rec/s

✅ Consumed 500,000 records in 20.2s


## 4.3 Process & Clean

In [10]:
df = pd.DataFrame(records)
del records
gc.collect()

print(f'Raw: {len(df):,} rows')

# Type conversions
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Quantity']    = pd.to_numeric(df['Quantity'],   errors='coerce')
df['UnitPrice']   = pd.to_numeric(df['UnitPrice'],  errors='coerce')
df['CustomerID']  = pd.to_numeric(df['CustomerID'], errors='coerce').astype('Int64')

# Filter
df = df.dropna(subset=['StockCode', 'InvoiceDate'])
df = df[df['Quantity']  > 0]
df = df[df['UnitPrice'] > 0]
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print(f'Clean: {len(df):,} rows')

Raw: 500,000 rows
Clean: 500,000 rows


## 4.4 Feature Engineering

In [11]:
df['Hour']        = df['InvoiceDate'].dt.hour
df['DayOfWeek']   = df['InvoiceDate'].dt.dayofweek
df['DayName']     = df['InvoiceDate'].dt.day_name()
df['Month']       = df['InvoiceDate'].dt.month
df['Year']        = df['InvoiceDate'].dt.year
df['WeekOfYear']  = df['InvoiceDate'].dt.isocalendar().week.astype(int)
df['Date']        = df['InvoiceDate'].dt.date.astype(str)
df['ProcessedAt'] = pd.Timestamp.now().isoformat()

print(f'Features added. Columns: {df.columns.tolist()}')
print(f'Shape: {df.shape}')

Features added. Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Hour', 'DayOfWeek', 'DayName', 'Month', 'Year', 'WeekOfYear', 'Date', 'ProcessedAt']
Shape: (500000, 17)


## 4.5 Write Transactions → Delta Lake

In [12]:
# Prepare for Delta
df_write = df.copy()
df_write['InvoiceDate'] = df_write['InvoiceDate'].astype(str)
df_write['CustomerID']  = df_write['CustomerID'].astype(str)

# Convert Int64 nullable columns
for col in df_write.select_dtypes(include='Int64').columns:
    df_write[col] = df_write[col].astype(float)

table = pa.Table.from_pandas(df_write, preserve_index=False)

print(f'Writing {len(df_write):,} records to Delta Lake...')
write_deltalake(DELTA_TX, table, mode='overwrite', storage_options=STORAGE)
print(f'✅ Transactions → {DELTA_TX}')

del df_write, table
gc.collect()

Writing 500,000 records to Delta Lake...
✅ Transactions → s3://retail-v2/delta/transactions


10

## 4.6 Verify Delta Lake

In [15]:
dt = DeltaTable(DELTA_TX, storage_options=STORAGE)
print(f'✅ Delta Table verified.')
print(f'   Version : {dt.version()}')

# Quick sample
sample = dt.to_pandas().head(5)
print(f'   Rows    : {len(dt.to_pandas()):,}')
print(f'\nSample:')
print(sample[['InvoiceNo','StockCode','Quantity','Revenue','Country','Date']].to_string(index=False))
del sample

✅ Delta Table verified.
   Version : 0
   Rows    : 500,000

Sample:
InvoiceNo StockCode  Quantity   Revenue        Country       Date
   495945     84839       2.0 13.763011 United Kingdom 2010-01-27
   495945     22333       4.0  6.457654 United Kingdom 2010-01-27
   495945     40003      12.0 15.452402 United Kingdom 2010-01-28
   495945     21466       4.0 14.925352 United Kingdom 2010-01-27
   495945     21472       5.0 18.338506 United Kingdom 2010-01-27


## 4.7 Feature Aggregation Table (for ML)

In [16]:
df_feat = (
    df.groupby(['StockCode', 'Country', 'Year', 'Month', 'WeekOfYear', 'DayOfWeek'])
    .agg(
        TotalQuantity   = ('Quantity',    'sum'),
        TotalRevenue    = ('Revenue',     'sum'),
        AvgUnitPrice    = ('UnitPrice',   'mean'),
        NumInvoices     = ('InvoiceNo',   'nunique'),
        UniqueCustomers = ('CustomerID',  'nunique'),
        Description     = ('Description', 'first')
    )
    .reset_index()
)
df_feat = df_feat[df_feat['TotalQuantity'] > 0]

print(f'Feature records: {len(df_feat):,}')

# Write
feat_table = pa.Table.from_pandas(df_feat, preserve_index=False)
write_deltalake(DELTA_FEAT, feat_table, mode='overwrite', storage_options=STORAGE)
print(f'✅ Features → {DELTA_FEAT}')

Feature records: 272,535
✅ Features → s3://retail-v2/delta/features


## 4.8 Dashboard Aggregations

In [17]:
# Daily revenue
daily = (
    df.groupby('Date')
    .agg(DailyRevenue=('Revenue','sum'), DailyTx=('InvoiceNo','nunique'), DailyCust=('CustomerID','nunique'))
    .reset_index().sort_values('Date')
)

# Country revenue
country = (
    df.groupby('Country')
    .agg(TotalRevenue=('Revenue','sum'), Transactions=('InvoiceNo','count'), Customers=('CustomerID','nunique'))
    .reset_index().sort_values('TotalRevenue', ascending=False).head(15)
)

# Top products
top_prods = (
    df.groupby(['StockCode','Description'])
    .agg(TotalQty=('Quantity','sum'), TotalRevenue=('Revenue','sum'))
    .reset_index().sort_values('TotalRevenue', ascending=False).head(20)
)

# Hourly heatmap data
heatmap = (
    df.groupby(['DayName','Hour'])['Revenue'].sum().reset_index()
)

# Monthly
df['YearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')
monthly = (
    df.groupby('YearMonth').agg(Revenue=('Revenue','sum'), Qty=('Quantity','sum'))
    .reset_index().sort_values('YearMonth')
)

# Write all
write_deltalake(f'{DELTA_AGG}/daily',        pa.Table.from_pandas(daily,     preserve_index=False), mode='overwrite', storage_options=STORAGE)
write_deltalake(f'{DELTA_AGG}/country',      pa.Table.from_pandas(country,   preserve_index=False), mode='overwrite', storage_options=STORAGE)
write_deltalake(f'{DELTA_AGG}/top_products', pa.Table.from_pandas(top_prods, preserve_index=False), mode='overwrite', storage_options=STORAGE)
write_deltalake(f'{DELTA_AGG}/heatmap',      pa.Table.from_pandas(heatmap,   preserve_index=False), mode='overwrite', storage_options=STORAGE)
write_deltalake(f'{DELTA_AGG}/monthly',      pa.Table.from_pandas(monthly,   preserve_index=False), mode='overwrite', storage_options=STORAGE)

print(f'✅ All aggregation tables written.')
print(f'   Daily:        {len(daily):,} rows')
print(f'   Country:      {len(country):,} rows')
print(f'   Top products: {len(top_prods):,} rows')
print(f'   Heatmap:      {len(heatmap):,} rows')
print(f'   Monthly:      {len(monthly):,} rows')

✅ All aggregation tables written.
   Daily:        457 rows
   Country:      15 rows
   Top products: 20 rows
   Heatmap:      168 rows
   Monthly:      16 rows


## 4.9 Delta Lake Time Travel

In [18]:
dt = DeltaTable(DELTA_TX, storage_options=STORAGE)
print(f'Version: {dt.version()}')
print('\nHistory:')
for h in dt.history():
    print(f"  v{h['version']} | {h['timestamp']} | {h['operation']}")

Version: 0

History:
  v0 | 1777115964956 | WRITE


## 4.10 Summary

In [19]:
print('='*55)
print('NOTEBOOK 4 COMPLETE')
print('='*55)
print(f'  Kafka consumed  : {len(df):,} records')
print(f'  Feature rows    : {len(df_feat):,}')
print(f'  Countries       : {df["Country"].nunique()}')
print(f'  Products        : {df["StockCode"].nunique()}')
print(f'  Total revenue   : £{df["Revenue"].sum():,.2f}')
print()
print('Delta tables on MinIO (retail-v2 bucket):')
print(f'  {DELTA_TX}')
print(f'  {DELTA_FEAT}')
print(f'  {DELTA_AGG}/daily')
print(f'  {DELTA_AGG}/country')
print(f'  {DELTA_AGG}/top_products')
print(f'  {DELTA_AGG}/heatmap')
print(f'  {DELTA_AGG}/monthly')
print()
print('View: http://localhost:9001 → retail-v2 bucket')
print('Next: Notebook 5 — Spark Batch Processing')

NOTEBOOK 4 COMPLETE
  Kafka consumed  : 500,000 records
  Feature rows    : 272,535
  Countries       : 40
  Products        : 4045
  Total revenue   : £10,881,574.77

Delta tables on MinIO (retail-v2 bucket):
  s3://retail-v2/delta/transactions
  s3://retail-v2/delta/features
  s3://retail-v2/delta/aggregations/daily
  s3://retail-v2/delta/aggregations/country
  s3://retail-v2/delta/aggregations/top_products
  s3://retail-v2/delta/aggregations/heatmap
  s3://retail-v2/delta/aggregations/monthly

View: http://localhost:9001 → retail-v2 bucket
Next: Notebook 5 — Spark Batch Processing
